# Notebook 2: Explore Relevant Projects in the GHTorrent Database

By this point, you should have the Gold Standard and GHTorrent 2004 dump loaded into MySQL. Since both datasets share the same comment IDs, we can join them to add contextual columns (e.g., project, author, timestamp) to the Gold Standard's three columns (`ID`, `polarity`, `text`).

But, before we create the contextualized Github Gold Standard dataset, we need to understand what we're working with and which projects have relevant sentiment-labeled comments that we want to use to be compatible with Kaiaulu

How are the 7,122 IDs split between commit comments and PR comments? Which projects show up the most? And are these comments reachable from canonical (non-fork) repos, or contained in forks?

The answers to these questions inform how we handle the data and which projects we target when generating project config files in Notebook 3.

### Step 1: Import dependencies and connect to MySQL

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_rows', None)

In [ ]:
MYSQL_HOST     = "localhost"
MYSQL_PORT     = 3306
MYSQL_USER     = "root"
MYSQL_PASSWORD = "ADD_YOUR_PASSWORD_HERE"
MYSQL_DB       = "github" # name of the database where GHTorrent was loaded

engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"
)
print("Connected to MySQL.")

### Check 1: How are the sentiment comments distributed?

GHTorrent stores two kinds of GitHub comments: commit comments (discussions on a specific commit) and PR inline comments (left on a line of code in a pull request).

To understand what commit comments look like, [here](https://github.com/openssl/openssl/commit/4817504d069b4c5082161b02a22116ad75f822b1#commitcomment-5942359) are examples of commit comments under a commit that introduced a popular software vulnerability. To understand what PR inline comments look like, refer to the [GitHub Pull Requests Cheatsheet](https://github.com/sailuh/kaiaulu_cheatsheet/blob/main/cheatsheets/github-comments-cheatsheet.pdf).

The Gold Standard includes both types. The same `ID` maps to `comment_id` in both `commit_comments` and `pull_request_comments`. So, the first thing to figure out is which table each sentiment ID lands in. Some IDs appear in both tables (overlap = 85), meaning a small number of comments were captured under both endpoints in GHTorrent. The total unique IDs should sum to 7,122.

Expected values:
- Commit comment matches: ~4,317
- PR comment matches: ~2,890
- Overlap (both): ~85
- Commit-only: 4,232 | PR-only: 2,805 | Total unique: 7,122

In [ ]:
with engine.connect() as con:
    commit_count = pd.read_sql(text("""
        SELECT COUNT(*) AS commit_comment_matches
        FROM comment_sentiment s
        INNER JOIN commit_comments cc ON s.ID = cc.comment_id;
    """), con).iloc[0, 0]

    pr_count = pd.read_sql(text("""
        SELECT COUNT(*) AS pr_comment_matches
        FROM comment_sentiment s
        INNER JOIN pull_request_comments prc ON s.ID = prc.comment_id;
    """), con).iloc[0, 0]

    overlap = pd.read_sql(text("""
        SELECT COUNT(*) AS overlap
        FROM comment_sentiment s
        INNER JOIN commit_comments cc ON s.ID = cc.comment_id
        INNER JOIN pull_request_comments prc ON s.ID = prc.comment_id;
    """), con).iloc[0, 0]

commit_only  = commit_count - overlap
pr_only      = pr_count - overlap
total_unique = commit_only + pr_only + overlap

summary = pd.DataFrame({
    'Category': ['Commit matches', 'PR matches', 'Overlap (both)', 'Commit-only', 'PR-only', 'Total unique'],
    'Count':    [commit_count, pr_count, overlap, commit_only, pr_only, total_unique],
    'Expected': [4317, 2890, 85, 4232, 2805, 7122]
})
display(summary)

if total_unique == 7122:
    print("PASS: total unique IDs = 7122.")
else:
    print(f"WARNING: total unique IDs = {total_unique}, expected 7122.")

### Check 2: Which projects have the most labeled commit comments?

Let's see which projects' commit comments are most heavily represented in the Gold Standard. This is a preview of which projects we'll be generating Kaiaulu config files for in Notebook 3.

In [ ]:
query_check2 = """
SELECT
    p.id AS project_id,
    p.name AS project_name,
    p.url AS project_url,
    COUNT(DISTINCT s.ID) AS labeled_comment_count
FROM projects p
INNER JOIN commits c ON p.id = c.project_id
INNER JOIN commit_comments cc ON c.id = cc.commit_id
INNER JOIN comment_sentiment s ON cc.comment_id = s.ID
GROUP BY p.id, p.name, p.url
ORDER BY labeled_comment_count DESC;
"""

with engine.connect() as con:
    check2 = pd.read_sql(text(query_check2), con)

print(f"Projects with sentiment-labeled commit comments: {len(check2)}")
display(check2)

### Check 3: Which projects have the most labeled PR comments?

Now, let's do the same project ranking for the pull request inline comments most heavily represented in the Gold Standard.

In [ ]:
query_check3 = """
SELECT
    p.id AS project_id,
    p.name AS project_name,
    p.url AS project_url,
    COUNT(DISTINCT s.ID) AS labeled_comment_count
FROM projects p
INNER JOIN pull_requests pr ON p.id = pr.base_repo_id
INNER JOIN pull_request_comments prc ON pr.id = prc.pull_request_id
INNER JOIN comment_sentiment s ON prc.comment_id = s.ID
GROUP BY p.id, p.name, p.url
ORDER BY labeled_comment_count DESC;
"""

with engine.connect() as con:
    check3 = pd.read_sql(text(query_check3), con)

print(f"Projects with sentiment-labeled PR comments: {len(check3)}")
display(check3)

### Check 4: Are the labeled comments reachable from canonical repos?

Projects on GitHub get forked all the time. Since forks share commit history with their upstream, the same comment IDs can appear under multiple projects in GHTorrent. This matters for Notebook 3 (config files generation). We want to know: if we only generate Kaiaulu configs for canonical (non-fork) repos, how much labeled data will we miss? The purpose of this query is to inform our coverage strategy going into Notebook 3.

Expected values:
- `canonical_only`: ~4,555
- `fork_only`: ~569 (these will be missed when targeting canonical repos only)
- `both_sides`: ~2,083
- Fork only rate: ~7.9%
- Canonical reachable rate: ~92.1%

From these values, we can see that ~92.1% of comments are reachable from canonical repos. The ~7.9% that are fork-only will be skipped when we generate project config files in Notebook 3. This is an acceptable tradeoff. We document it here so the limitation is visible.

In [ ]:
query_check4 = """
WITH RECURSIVE project_root AS (
    SELECT p.id AS project_id, p.id AS root_id
    FROM projects p
    WHERE p.forked_from IS NULL
    UNION ALL
    SELECT c.id AS project_id, pr.root_id
    FROM projects c
    JOIN project_root pr ON c.forked_from = pr.project_id
),
comment_project_rows AS (
    SELECT cs.ID AS comment_id, c.project_id, 'commit_comment' AS source_tag
    FROM comment_sentiment cs
    JOIN commit_comments cc ON cs.ID = cc.comment_id
    JOIN commits c ON cc.commit_id = c.id
    UNION ALL
    SELECT cs.ID AS comment_id, pr.base_repo_id AS project_id, 'pr_comment' AS source_tag
    FROM comment_sentiment cs
    JOIN pull_request_comments prc ON cs.ID = prc.comment_id
    JOIN pull_requests pr ON prc.pull_request_id = pr.id
    UNION ALL
    SELECT cs.ID AS comment_id, pr.head_repo_id AS project_id, 'pr_comment' AS source_tag
    FROM comment_sentiment cs
    JOIN pull_request_comments prc ON cs.ID = prc.comment_id
    JOIN pull_requests pr ON prc.pull_request_id = pr.id
),
labeled AS (
    SELECT
        cpr.comment_id,
        cpr.source_tag,
        pr.root_id,
        (cpr.project_id = pr.root_id) AS is_canonical
    FROM comment_project_rows cpr
    JOIN project_root pr ON pr.project_id = cpr.project_id
),
comment_flags AS (
    SELECT
        root_id, source_tag, comment_id,
        MAX(CASE WHEN is_canonical THEN 1 ELSE 0 END) AS has_canonical,
        MAX(CASE WHEN NOT is_canonical THEN 1 ELSE 0 END) AS has_fork
    FROM labeled
    GROUP BY root_id, source_tag, comment_id
),
global_counts AS (
    SELECT
        COUNT(*) AS mapped_comment_ids,
        SUM(CASE WHEN has_canonical = 1 AND has_fork = 0 THEN 1 ELSE 0 END) AS canonical_only,
        SUM(CASE WHEN has_canonical = 0 AND has_fork = 1 THEN 1 ELSE 0 END) AS fork_only,
        SUM(CASE WHEN has_canonical = 1 AND has_fork = 1 THEN 1 ELSE 0 END) AS both_sides
    FROM comment_flags
)
SELECT
    canonical_only,
    fork_only,
    both_sides,
    ROUND(100 * fork_only / NULLIF(mapped_comment_ids, 0), 2) AS fork_only_pct,
    ROUND(100 * (canonical_only + both_sides) / NULLIF(mapped_comment_ids, 0), 2) AS canonical_reachable_pct
FROM global_counts;
"""

with engine.connect() as con:
    check4 = pd.read_sql(text(query_check4), con)

print("Canonical vs fork accessibility summary:")
print(f"  canonical_only: {check4['canonical_only'].iloc[0]} (expected ~4555)")
print(f"  fork_only: {check4['fork_only'].iloc[0]} (expected ~569)")
print(f"  both_sides: {check4['both_sides'].iloc[0]} (expected ~2083)")
print(f"  fork_only %: {check4['fork_only_pct'].iloc[0]}% (expected ~7.9%)")
print(f"  canonical_reachable %: {check4['canonical_reachable_pct'].iloc[0]}% (expected ~92.1%)")
display(check4)

### Step 5: Build the contextualized dataset

Now that we know which projects have sentiment-labeled comments and how they map across tables, we can build the contextualized dataset.

The Gold Standard currently has three columns (`ID`, `polarity`, `text`). We're going to add six more from GHTorrent:

1. `created_at` - Comment timestamp
2. `author_login` - Author username
3. `author_name` - Author First Name & Last Name
4. `author_email` - Author email
5. `owner` - Project owner
6. `repo` - Project repo name

In [ ]:
# Pull context for commit comments
commit_context_sql = """
SELECT
    cs.ID            AS comment_id,
    cs.Polarity      AS polarity,
    cs.Text          AS text,
    cc.created_at    AS created_at,
    u.login          AS author_login,
    u.name           AS author_name,
    u.email          AS author_email,
    u_owner.login    AS owner,
    p.name           AS repo
FROM comment_sentiment cs
JOIN commit_comments cc  ON cs.ID = cc.comment_id
JOIN users u             ON cc.user_id = u.id
JOIN commits c           ON cc.commit_id = c.id
JOIN projects p          ON c.project_id = p.id
JOIN users u_owner       ON p.owner_id = u_owner.id
"""

# Pull context for PR comments
pr_context_sql = """
SELECT
    cs.ID           AS comment_id,
    cs.Polarity     AS polarity,
    cs.Text         AS text,
    prc.created_at  AS created_at,
    u.login         AS author_login,
    u.name          AS author_name,
    u.email         AS author_email,
    u_owner.login   AS owner,
    p.name          AS repo
FROM comment_sentiment cs
JOIN pull_request_comments prc ON cs.ID = CAST(prc.comment_id AS UNSIGNED)
JOIN users u                   ON prc.user_id = u.id
JOIN pull_requests pr          ON prc.pull_request_id = pr.id
JOIN projects p                ON pr.base_repo_id = p.id
JOIN users u_owner             ON p.owner_id = u_owner.id
"""

with engine.connect() as con:
    commit_ctx = pd.read_sql(text(commit_context_sql), con)
    pr_ctx     = pd.read_sql(text(pr_context_sql), con)

# Deduplicate within each source: keep first match per comment_id
commit_ctx = commit_ctx.drop_duplicates(subset='comment_id', keep='first')
pr_ctx     = pr_ctx.drop_duplicates(subset='comment_id', keep='first')

# Merge: prefer commit comment rows; fill in PR-only rows for IDs not in commit set
commit_ids   = set(commit_ctx['comment_id'])
pr_only      = pr_ctx[~pr_ctx['comment_id'].isin(commit_ids)]
contextualized = (
    pd.concat([commit_ctx, pr_only], ignore_index=True)
    .sort_values('comment_id')
    .reset_index(drop=True)
)

print(f"Total rows: {len(contextualized)}  (expected 7122)")
print(f"  From commit comments: {len(commit_ctx)}")
print(f"  From PR comments only: {len(pr_only)}")

### Step 6: Compare original vs. contextualized dataset

Let's see a quick before/after to see what columns we added. The original Gold Standard has three columns, while the new contextualized version we created has ten.

In [ ]:
with engine.connect() as con:
    original = pd.read_sql(text("SELECT ID, Polarity, Text FROM comment_sentiment LIMIT 5;"), con)

print("Original GitHub Gold Standard (first 5 rows):")
display(original)

print("\nContextualized dataset with additional GHTorrent columns (first 5 rows):")
display(contextualized.head())

null_emails = contextualized['author_email'].isna().sum()
print(f"\nNote: {null_emails} of {len(contextualized)} rows have a NULL author_email ({round(100*null_emails/len(contextualized), 1)}%). GitHub stopped exposing emails in the API, so this is expected.")